In [1]:
!pip install -q -U transformers trl peft datasets bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 113.8 MB/s eta 0:00:0000:01


In [ ]:
import torch
import json
import os
import shutil
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer, DPOConfig
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login to Hugging Face
login(token=hf_token)

print("Libraries imported and login successful.")

Libraries imported and login successful.


In [3]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# 4-bit configuration with optimizations
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True, # Saves extra memory
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    # attn_implementation="sdpa", # Uses Scaled Dot Product Attention for speed/memory
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print("Model loaded with 4-bit quantization.")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Model loaded with 4-bit quantization.


In [4]:
lora_config = LoraConfig(
    r=8,  # Reduced from 16 to 8 to save memory
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
print("LoRA config initialized.")

LoRA config initialized.


In [5]:
def format_dpo_dataset(triplets_file):
    if not os.path.exists(triplets_file):
        # Create dummy data if file is missing just to test script
        print("File not found, creating dummy data...")
        data = [{"prompt": "Hello", "chosen": "Hi there!", "rejected": "Go away"}]
    else:
        with open(triplets_file, "r") as f:
            data = json.load(f)

    formatted_data = []
    for item in data:
        formatted_data.append({
            "prompt": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{item['prompt']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
            "chosen": item["chosen"] + "<|eot_id|>",
            "rejected": item["rejected"] + "<|eot_id|>"
        })

    return Dataset.from_list(formatted_data)

# Update path if you uploaded it via Kaggle Datasets (e.g., "/kaggle/input/your-dataset/pi_...json")
train_dataset = format_dpo_dataset("/kaggle/input/datasets/vidyullatas/final-enterprise-ds/rdpo_dataset_final.json")
print(f"Dataset loaded: {len(train_dataset)} samples.")

Dataset loaded: 798 samples.


In [7]:
# 1. Setup the Config
# All parameters INCLUDING lengths must go here in the latest TRL
dpo_config = DPOConfig(
    output_dir="./pi-dpo-llama3-adapter",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16, # Effective batch size of 16
    gradient_checkpointing=True,    # Crucial for preventing OOM
    learning_rate=1e-4,
    logging_steps=5,
    save_steps=25,
    bf16=True,                      # Use bfloat16 for memory efficiency
    optim="paged_adamw_8bit",       # Saves ~2GB VRAM
    warmup_steps=20,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    beta=0.05,
    max_length=256                 # This MUST be here in newer TRL
)

# 2. Initialize Trainer
# Use 'processing_class' (new name for tokenizer)
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None, 
    args=dpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer, 
    peft_config=lora_config,
)

print("DPOTrainer ready with Gradient Checkpointing.")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/798 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/798 [00:00<?, ? examples/s]

DPOTrainer ready with Gradient Checkpointing.


In [8]:
print("Starting Training...")
dpo_trainer.train()

# Save adapter
final_adapter_path = "./pi-dpo-final-adapter"
dpo_trainer.save_model(final_adapter_path)

# Zip for download
shutil.make_archive('pi-dpo-adapter', 'zip', final_adapter_path)
print(f"Training finished. Adapter saved and zipped.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Starting Training...


Step,Training Loss
5,0.693177
10,0.681733
15,0.618991
20,0.493335
25,0.316948
30,0.280359
35,0.149719
40,0.091666
45,0.093836
50,0.170567


Training finished. Adapter saved and zipped.


In [18]:
c100_adapter_path = "./kaggle/working/pi-dpo-llama3-adapter/checkpoint-100"
dpo_trainer.save_model(c100_adapter_path)

# Zip for download
shutil.make_archive('bestacc-dpo-adapter', 'zip', c100_adapter_path)
print(f"Training finished. Adapter saved and zipped.")

Training finished. Adapter saved and zipped.


In [12]:
import zipfile
import os
import torch
from peft import PeftModel

# 1. Unzip the adapter (Python native way)
# zip_path = "/kaggle/working/pi-dpo-adapter.zip"
extract_path = "/kaggle/working/pi-dpo-llama3-adapter/checkpoint-75"

# print(f"Unzipping adapter from {zip_path}...")
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)
# print("Unzipping complete!")

# 2. Inject the custom DPO adapter into your already-loaded base model
print(f"Injecting custom DPO adapter from {extract_path}...")
dpo_model = PeftModel.from_pretrained(model, extract_path)
print("✅ Adapter successfully injected! Model is now intrinsically robust.")

# 3. The Demo Function (Fixed the parameter!)
def ask_dpo_model(messy_question):
    # Use Llama 3's required chat formatting
    messages =[{"role": "user", "content": messy_question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate the answer using the DPO model
    outputs = dpo_model.generate(
        **inputs, 
        max_new_tokens=100, 
        temperature=0.0, 
        do_sample=False, 
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Slice off the prompt to only show the generated assistant text
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Robust splitting to isolate just the AI's answer
    if "assistant\n\n" in full_response:
        return full_response.split("assistant\n\n")[-1].strip()
    else:
        return full_response # Fallback just in case

# --- 4. THE LIVE PROFESSOR DEMO ---
print("\n" + "="*60)
print("🤖 LIVE DEMO: rDPO-Llama-3 vs. Semantic Noise")
print("="*60)

# Test 1: Formal (The "Good" user)
q1 = "What is the remaining balance of sick leave I have accumulated so far this year?"
print(f"\n[Formal Input]: {q1}")
print(f"🎯 [DPO Output]: {ask_dpo_model(q1)}")

# Test 2: Highly degraded / messy (The "Chaotic" user)
q2 = "how many sikk leave dos i got left 4 this year?"
print(f"\n[Messy Input] : {q2}")
print(f"🎯[DPO Output]: {ask_dpo_model(q2)}")

# Test 3: Completely conversational (The "Slang" user)
q3 = "i wanna know how many sickdays i got lef in this year"
print(f"\n[Slang Input] : {q3}")
print(f"🎯[DPO Output]: {ask_dpo_model(q3)}")

Injecting custom DPO adapter from /kaggle/working/pi-dpo-llama3-adapter/checkpoint-75...


Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Adapter successfully injected! Model is now intrinsically robust.

🤖 LIVE DEMO: rDPO-Llama-3 vs. Semantic Noise

[Formal Input]: What is the remaining balance of sick leave I have accumulated so far this year?


Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎯 [DPO Output]: You can check your accumulated sick leave balance by logging into the HR portal, clicking on the "My Leave" tab, and selecting "Sick Leave Balance" from the drop-down menu. Your current balance will be displayed. HR will also send a separate email to your manager and HR on your first working day each year (January 1st) with your sick leave balance reset.

[Messy Input] : how many sikk leave dos i got left 4 this year?


Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎯[DPO Output]: You have carried forward 2 Sick Leaves from 2022, which are reset on January 1st every year. You can check your carried forward balance by logging into the Benefits > Leave > Carry Forward Balance portal in the HR portal.

[Slang Input] : i wanna know how many sickdays i got lef in this year
🎯[DPO Output]: To check your remaining sick leave balance, you can send an email to hr@yoursecondstory.com with the subject "Sick Leave Balance" and your employee ID as the reference. HR will respond with your current balance via email.


In [14]:
# --- 5. ABLATION STUDY: BASE MODEL vs. rDPO MODEL ---
print("\n" + "="*60)
print("🔬 ABLATION STUDY: BASE MODEL vs. DPO ADAPTER")
print("="*60)

test_questions =[
    "How can I cancel a leave request that has already been approved?",
    "Is it possible to modify or withdraw my approved leave in the HR portal?",
    "What should I do if I want to cancel my previously approved time-off request"
]

for i, q in enumerate(test_questions):
    print(f"\n--- Test {i+1} ---")
    print(f"🗣️ [Input Query]: {q}")
    
    # 1. Use Llama-3's Native Chat Template!
    messages = [{"role": "user", "content": q}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(dpo_model.device)
    
    # Get the exact length of the input prompt so we can slice it off later
    input_length = inputs['input_ids'].shape[1] 
    
    # =================================================================
    # 1. BASE MODEL (Disable the custom DPO brain temporarily)
    # =================================================================
    with dpo_model.disable_adapter():
        base_outputs = dpo_model.generate(
            **inputs, 
            max_new_tokens=60, 
            temperature=0.0, 
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
        
        # PRO TRICK: Slice the output tensor to grab ONLY the new tokens
        base_new_tokens = base_outputs[0][input_length:]
        base_ans = tokenizer.decode(base_new_tokens, skip_special_tokens=True).strip()
        
        # Clean up any trailing newlines Base model might add
        base_ans = base_ans.split('\n')[0] 
        print(f"🛑 [Base Llama-3]: {base_ans}")
        
    # =================================================================
    # 2. rDPO MODEL (Re-enable the custom DPO brain)
    # =================================================================
    dpo_outputs = dpo_model.generate(
        **inputs, 
        max_new_tokens=60, 
        temperature=0.0, 
        do_sample=False, 
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Slice the output tensor to grab ONLY the new tokens
    dpo_new_tokens = dpo_outputs[0][input_length:]
    dpo_ans = tokenizer.decode(dpo_new_tokens, skip_special_tokens=True).strip()
    
    dpo_ans = dpo_ans.split('\n')[0]
    print(f"🎯[rDPO Llama-3]: {dpo_ans}")

Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔬 ABLATION STUDY: BASE MODEL vs. DPO ADAPTER

--- Test 1 ---
🗣️ [Input Query]: How can I cancel a leave request that has already been approved?


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🛑 [Base Llama-3]: The pesky situation of needing to cancel a leave request that's already been approved!


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

🛑 [Base Llama-3]: The convenience of online leave management!


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎯[rDPO Llama-3]: Yes, you can modify or withdraw an approved leave by logging into the HR portal, navigating to the "Leaves" tab, and clicking on "Modify/Cancel Existing Leave". Select the approved leave record you want to modify or cancel, and make the necessary changes or click "Cancel" to withdraw

--- Test 3 ---
🗣️ [Input Query]: What should I do if I want to cancel my previously approved time-off request


Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🛑 [Base Llama-3]: If you want to cancel a previously approved time-off request, you should follow the same process as when you initially requested the time off, but this time, you'll be requesting to cancel the request. Here are some steps you can follow:
🎯[rDPO Llama-3]: To cancel a previously approved time-off request, follow these steps:
